In [162]:
import random
import csv
from collections import Counter

# ----- talia -----
SUITS = ['S', 'H', 'D', 'C']  # piki, kiery, kara, trefle
RANKS = ['2', '3', '4', '5', '6', '7', '8', '9', 'T', 'J', 'Q', 'K', 'A']

DECK = [r + s for s in SUITS for r in RANKS]

# ----- HCP -----
HCP_VALUES = {
    'A': 4,
    'K': 3,
    'Q': 2,
    'J': 1
}

def calculate_hcp(hand):
    return sum(HCP_VALUES.get(card[0], 0) for card in hand)

# ----- długości kolorów -----
def suit_lengths(hand):
    cnt = Counter(card[1] for card in hand)
    return [cnt['S'], cnt['H'], cnt['D'], cnt['C']]

# ----- ręka zrównoważona -----
def rowny_sklad(d):
    d_sorted = sorted(d, reverse=True)
    return d_sorted in ([4,3,3,3], [4,4,3,2], [5,3,3,2])

def prawo20(d,pc):
  top2 = sorted(d, reverse=True)[:2]
  if pc + top2[0] + top2[1] >= 20:
    return True
  else:
    return False

# ----- Twoja funkcja w Pythonie -----
def co_otworzyc(dlugosc, pc):
    if pc >= 12 or prawo20(dlugosc, pc):
        if dlugosc[0] >= 5 and pc < 18:
            return "1S"
        elif dlugosc[1] >= 5 and pc < 18:
            return "1H"
        elif pc >= 15 and pc <= 17 and rowny_sklad(dlugosc):
            return "1NT"
        elif dlugosc[2] >= 4 and pc < 18:
            return "1D"
        else:
            return "1C"

    elif pc >= 6 and (
        (dlugosc[0] >= 5 and dlugosc[1] >= 4) or
        (dlugosc[1] >= 5 and dlugosc[0] >= 4)
    ):
        return "2C"

    elif pc >= 6 and (dlugosc[0] == 6 or dlugosc[1] == 6):
        return "2D"

    else:
        return "PASS"

# ----- generowanie jednej ręki -----
def deal_hand():
    return random.sample(DECK, 13)

# ----- główna funkcja -----
def generate_hands(n, filename="hands.csv"):
    with open(filename, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["hand", "S", "H", "D", "C", "HCP", "opening"])

        for _ in range(n):
            hand = deal_hand()

            d = suit_lengths(hand)
            pc = calculate_hcp(hand)
            opening = co_otworzyc(d, pc)

            writer.writerow([
                " ".join(hand),
                d[0], d[1], d[2], d[3],
                pc,
                opening
            ])

# ----- uruchomienie -----
if __name__ == "__main__":
    generate_hands(10000)

In [163]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib

# ----- wczytanie danych -----
df = pd.read_csv("hands.csv")

# ----- features -----
X = df[["S", "H", "D", "C", "HCP"]]

# ----- target -----
y = df["opening"]

# kodowanie etykiet
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ----- split -----
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# ----- model -----
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

# ----- trening -----
model.fit(X_train, y_train)

# ----- ewaluacja -----
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

# ----- zapis modelu -----
joblib.dump(model, "bridge_opening_model.pkl")
joblib.dump(le, "label_encoder.pkl")

              precision    recall  f1-score   support

          1C       1.00      1.00      1.00       256
          1D       1.00      1.00      1.00       204
          1H       0.99      0.98      0.98       130
         1NT       1.00      1.00      1.00        67
          1S       1.00      0.99      1.00       159
          2C       1.00      1.00      1.00        48
          2D       0.98      1.00      0.99        60
        PASS       1.00      1.00      1.00      1076

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



['label_encoder.pkl']

In [174]:
import random
import joblib
from collections import Counter

# ----- load model -----
model = joblib.load("bridge_opening_model.pkl")
le = joblib.load("label_encoder.pkl")

# ----- deck -----
SUITS = ['S', 'H', 'D', 'C']
RANKS = ['2','3','4','5','6','7','8','9','T','J','Q','K','A']
DECK = [r + s for s in SUITS for r in RANKS]

# ----- HCP -----
HCP_VALUES = {'A':4,'K':3,'Q':2,'J':1}

def hcp(hand):
    return sum(HCP_VALUES.get(card[0], 0) for card in hand)

def suit_count(hand):
    c = Counter(card[1] for card in hand)
    return [c['S'], c['H'], c['D'], c['C']]

def deal():
    return random.sample(DECK, 13)

def predict_opening(s, h, d, c, hcp_value):
    X = [[s, h, d, c, hcp_value]]
    pred = model.predict(X)[0]
    return le.inverse_transform([pred])[0]

def show_hand(hand):
    print("HAND:")
    print(" ".join(hand))

# ----- test -----
for _ in range(10):
  hand = deal()

  s, h, d, c = suit_count(hand)
  points = hcp(hand)

  opening = predict_opening(s, h, d, c, points)

  show_hand(hand)

  print("\nDługości kolorów:")
  print(f"S:{s}, H:{h}, D:{d}, C:{c}")

  print(f"\nHCP: {points}")
  print(f" MODEL OTWIERA: {opening}")

print("TAK")

HAND:
AD 5C 5H 7S 7C AH 4H TC 7H 2H JS JD JC

Długości kolorów:
S:2, H:5, D:2, C:4

HCP: 11
 MODEL OTWIERA: 1H
HAND:
AH 7D 2C KH 3C 2H JH 6H 4C 5H 7C 3D 8C

Długości kolorów:
S:0, H:6, D:2, C:5

HCP: 8
 MODEL OTWIERA: 2D
HAND:
KS 8S AC 7S TD 7D KD 2S 8H 3S 8C 4H 2C

Długości kolorów:
S:5, H:2, D:3, C:3

HCP: 10
 MODEL OTWIERA: PASS
HAND:
4D 6S 3S 2H AC 2C 3D JS 5H 5S TC 6D KC

Długości kolorów:
S:4, H:2, D:3, C:4

HCP: 8
 MODEL OTWIERA: PASS
HAND:
6C QH 9D 3H 4H 4C QD 5D 4S 7C JS TH 5H

Długości kolorów:
S:2, H:5, D:3, C:3

HCP: 5
 MODEL OTWIERA: PASS
HAND:
AH JS 9H 6S QS 2H 6D 9S 6H AC TC 7H TS

Długości kolorów:
S:5, H:5, D:1, C:2

HCP: 11
 MODEL OTWIERA: 1S
HAND:
AH 3S QS 2H TD 7C QD 4S JS 4H AD KD 8H

Długości kolorów:
S:4, H:4, D:4, C:1

HCP: 16
 MODEL OTWIERA: 1D
HAND:
7D 5D 3D 9H 2C QS 3S JD 2D 9D QC 6H 2H

Długości kolorów:
S:2, H:3, D:6, C:2

HCP: 5
 MODEL OTWIERA: PASS
HAND:
JC KD TH 6D 2D 8H QC 2S QS 3S 7D 4H 5S

Długości kolorów:
S:4, H:3, D:4, C:2

HCP: 8
 MODEL OTWIERA: P

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local